# Day 0C — Coding Practice for Fine-Tuning

The second programming lesson. Day 0 taught the building blocks; Day 0C uses them on real fine-tuning jobs: reading and writing dataset files, cleaning messy examples, splitting train/validation data, padding token batches, configuration objects and a loop that learns. Do Day 0 first.

**How to use this notebook in Google Colab**
- Click inside a grey code cell and press **Shift + Enter**. Run cells **in order** — later lessons use the data created in earlier ones.
- Outputs are already saved here so you can read them first. Your results should match exactly.
- The notebook creates small files (`notes.txt`, `raw_captions.jsonl`, a `data/` folder). See them with the **folder icon** on the left.
- No GPU needed. If anything gets confusing: **Runtime → Restart session**, then run from the top.

Time needed: about 3–4 hours.

---
## Lesson 1 — Files: saving and reading text

Training data lives in files, so the first skill is opening a file, writing to it and reading it back. `open(name, mode)` opens a file; mode `"w"` = write (replaces old content), `"r"` = read, `"a"` = append (add to the end). The `with` block closes the file for you when the indented lines finish.

In [1]:
note = "Fine-tuning practice\nStep 1: collect data\nStep 2: clean data\n"

with open("notes.txt", "w", encoding="utf-8") as f:
    f.write(note)

print("Saved notes.txt")

Saved notes.txt


**What happened:** `\n` is the invisible “new line” character. `as f` gives the open file a short name. `encoding="utf-8"` lets the file hold any language or symbol (Hindi, ₹, emoji); always include it.

In [2]:
with open("notes.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)
print("Characters:", len(content))

Fine-tuning practice
Step 1: collect data
Step 2: clean data

Characters: 61


**What happened:** `f.read()` returns the whole file as one piece of text. The 3 `\n` characters are counted in the length.

In [3]:
with open("notes.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(number, "->", line.strip())

1 -> Fine-tuning practice
2 -> Step 1: collect data
3 -> Step 2: clean data


**What happened:** Looping over a file gives one line at a time — perfect for datasets with one example per line. `enumerate(...)` adds a counter, so each round gives two values: the number and the line. `start=1` counts from 1 instead of 0. `.strip()` removes the `\n` at the end.

🔗 **Where you will use this:** `with` is the same keyword as `with torch.no_grad():` in Day 1: “do the indented lines inside this special setting, then tidy up”. Every training script opens dataset files exactly like this.

✏️ **Your turn:**
1. Change mode `"w"` to `"a"` in the first cell and run it twice. Read the file again. What happened?
2. Count the lines with `len(open("notes.txt").readlines())`.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 2 — JSON: dictionaries saved as text

A file can only store text, but our examples are dictionaries. **JSON** is the standard way to turn a dictionary into text and back. `json.dumps(d)` = dictionary → text (“dump to string”). `json.loads(text)` = text → dictionary (“load from string”).

In [4]:
import json

example = {
    "instruction": "Write an Instagram caption for a new product.",
    "input": "Product: millet cookies, Rs 249",
    "output": "Crunchy millet cookies, baked fresh. Rs 249 a box.",
}

line = json.dumps(example)
print(type(line))
print(line)

<class 'str'>
{"instruction": "Write an Instagram caption for a new product.", "input": "Product: millet cookies, Rs 249", "output": "Crunchy millet cookies, baked fresh. Rs 249 a box."}


**What happened:** The result is a `str` — plain text that looks like the dictionary. JSON always uses double quotes.

In [5]:
back = json.loads(line)
print(type(back))
print(back["input"])
print(json.dumps(back, indent=2))

<class 'dict'>
Product: millet cookies, Rs 249
{
  "instruction": "Write an Instagram caption for a new product.",
  "input": "Product: millet cookies, Rs 249",
  "output": "Crunchy millet cookies, baked fresh. Rs 249 a box."
}


**What happened:** `json.loads` gives a real dictionary again, so `back["input"]` works. `indent=2` prints it neatly over several lines — useful for reading, but datasets keep each example on **one** line.

🔗 **Where you will use this:** Every fine-tuning dataset you build is a file of JSON lines, one example per line. Hugging Face also saves model settings as JSON (`config.json`, which you will open on Day 3).

✏️ **Your turn:**
1. Add a key `"platform": "instagram"` to `example` and dump it again.
2. Try `json.loads("{'a': 1}")` (single quotes). Read the error: JSON requires double quotes.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 3 — JSONL: a whole dataset in one file

**JSONL** (“JSON Lines”) = one JSON dictionary per line. It is the most common format for fine-tuning data. Below we create a small *raw* dataset of captions. It deliberately contains problems you will clean up in later lessons: a duplicate, a missing field, an empty answer and messy spaces.

In [6]:
raw_records = [
    {"instruction": "Write an Instagram caption for a new product.", "input": "Product: millet cookies, Rs 249", "output": "Crunchy millet cookies, baked fresh this morning. Rs 249 a box."},
    {"instruction": "Write an Instagram caption for a customer review.", "input": "Customer: Priya. Product: brownies", "output": "Priya says our almond flour brownies beat regular ones. Thank you, Priya!"},
    {"instruction": "Write an Instagram caption for a behind-the-scenes post.", "input": "Scene: 6 am, kneading dough", "output": "6 am in our kitchen. Everything you order is baked the same morning."},
    {"instruction": "Write an Instagram caption for a new product.", "input": "Product: millet cookies, Rs 249", "output": "Crunchy millet cookies, baked fresh this morning. Rs 249 a box."},
    {"instruction": "Write an Instagram caption for a festival offer.", "input": "Festival: Diwali. Box of 4 treats, Rs 799"},
    {"instruction": "Write an Instagram caption for a restock.", "input": "Product: banana bread, 30 loaves", "output": "   "},
    {"instruction": "  Write an Instagram caption for a restock.  ", "input": "Product: walnut brownies, Saturday", "output": "Walnut brownies are back this Saturday.   Reserve yours in our DMs."},
    {"instruction": "Write an Instagram caption answering a question.", "input": "Question: Is gluten free only for coeliacs?", "output": "For coeliacs it is essential. Many others choose it because it suits them. Either way, our bakes taste great first."},
    {"instruction": "Write an Instagram caption for a festival offer.", "input": "Festival: Lohri. Til-gur cookies, Rs 199", "output": "Lohri special: til-gur cookies, gluten free and full of warmth. Rs 199 a pack."},
]

with open("raw_captions.jsonl", "w", encoding="utf-8") as f:
    for record in raw_records:
        f.write(json.dumps(record) + "\n")

print("Wrote", len(raw_records), "records")

Wrote 9 records


**What happened:** A list of 9 dictionaries. The loop turns each into one line of JSON text and adds `\n` so the next record starts on a new line. Can you already spot the problems? (Records 3, 4, 5 and 6 — counting from 0.)

In [7]:
records = []
with open("raw_captions.jsonl", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(len(records), "records loaded")
print(records[0]["output"])

9 records loaded
Crunchy millet cookies, baked fresh this morning. Rs 249 a box.


**What happened:** Reading is the reverse: for every line, `json.loads` it and add the dictionary to a list. This 4-line pattern is the heart of every dataset loader.

🔗 **Where you will use this:** On Day 4 you will load files like this with Hugging Face: `load_dataset("json", data_files="train.jsonl")`. It does the same loop for you, just faster and for millions of lines.

✏️ **Your turn:**
1. Print `records[4]`. Which key is missing?
2. Add a 10th record about a new product of your own and re-run both cells.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 4 — try / except: surviving bad data

Real data files contain broken lines. Without protection, one bad line crashes the whole program. `try:` runs some code; if a specific error happens, Python jumps to `except:` instead of stopping.

In [8]:
with open("raw_captions.jsonl", "a", encoding="utf-8") as f:
    f.write('{"instruction": "Write a caption", "output": "Oops, the closing bracket is missing"\n')

print("Added one broken line")

Added one broken line


**What happened:** Mode `"a"` adds to the end of the file. The new line is missing its closing `}`, so it is not valid JSON — just like a line damaged by a bad copy-paste.

In [9]:
bad = '{"instruction": "Write a caption", "output": "Oops"'

try:
    json.loads(bad)
    print("This line never runs")
except json.JSONDecodeError as error:
    print("Could not read it:", error)

print("The program kept going")

Could not read it: Expecting ',' delimiter: line 1 column 52 (char 51)
The program kept going


**What happened:** `json.loads` failed, so Python skipped the rest of the `try` block and ran the `except` block. `as error` stores the error message. Then the program carried on normally.

In [10]:
def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Skipped line {number}: not valid JSON")
    return records

records = load_jsonl("raw_captions.jsonl")
print(len(records), "records loaded")

Skipped line 10: not valid JSON
9 records loaded


**What happened:** A reusable loader. `continue` means “skip the rest of this round and go to the next line” — here it skips blank lines. Bad lines are reported with their line number and skipped. You will reuse this exact function.

🔗 **Where you will use this:** Training scripts use `try/except` to catch problems like running out of GPU memory (`torch.cuda.OutOfMemoryError`) and print a helpful message instead of crashing (Day 18).

✏️ **Your turn:**
1. Run the old 4-line loader from Lesson 3 on the file now. You will see the crash that `load_jsonl` avoids.
2. Add a blank line to the file (append `"\n"`) and check that `load_jsonl` still reports 9 records.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 5 — Looking inside records: .keys(), .get() and None

Before cleaning data, inspect it. Dictionaries have methods for this. `.keys()` lists the keys. `.get(key)` looks up a value **without crashing** if the key is missing: it returns `None` (Python’s word for “nothing”), or a default you choose.

In [11]:
first = records[0]
print(list(first.keys()))
print(first.get("output"))
print(records[4].get("output"))
print(records[4].get("output", "MISSING"))

['instruction', 'input', 'output']
Crunchy millet cookies, baked fresh this morning. Rs 249 a box.
None
MISSING


**What happened:** `records[4]` has no `output` key. `records[4]["output"]` would crash with a KeyError; `.get` returns `None` instead, or `"MISSING"` when we give a default.

In [12]:
for i, record in enumerate(records):
    has_output = "output" in record
    print(i, has_output, record["instruction"][:45])

0 True Write an Instagram caption for a new product.
1 True Write an Instagram caption for a customer rev
2 True Write an Instagram caption for a behind-the-s
3 True Write an Instagram caption for a new product.
4 False Write an Instagram caption for a festival off
5 True Write an Instagram caption for a restock.
6 True   Write an Instagram caption for a restock.  
7 True Write an Instagram caption answering a questi
8 True Write an Instagram caption for a festival off


**What happened:** `"output" in record` is True/False: does this dictionary have that key? `[:45]` slices the first 45 characters so each line stays short. Record 6 shows its leading spaces.

🔗 **Where you will use this:** Hugging Face models and datasets are often inspected the same way: `ds[0]` shows one example, `ds.column_names` lists the keys, `model.config.get(...)` reads a setting.

✏️ **Your turn:**
1. Print `records[5]`. What is wrong with its output?
2. Use `.get("platform", "instagram")` on a record. Why does it print `instagram`?

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 6 — Cleaning text with string methods

Text from real posts has extra spaces, line breaks and inconsistent formatting. Small mistakes like these are learned by the model too. A few string methods fix most of it. `repr()` prints text with its hidden characters visible, which makes spaces easy to see.

In [13]:
messy = "  Walnut brownies are back this Saturday.   Reserve yours in our DMs.  "

print(repr(messy))
print(repr(messy.strip()))
print(repr(" ".join(messy.split())))
print(messy.lower().count("brownies"))
print("DMs" in messy, messy.strip().endswith("."))

'  Walnut brownies are back this Saturday.   Reserve yours in our DMs.  '
'Walnut brownies are back this Saturday.   Reserve yours in our DMs.'
'Walnut brownies are back this Saturday. Reserve yours in our DMs.'
1
True True


**What happened:** `.strip()` removes spaces at both ends. `.split()` breaks text into words at any amount of space, and `" ".join(...)` glues them back with exactly one space — together they remove all extra spaces. `.lower()` makes lowercase; `.count()` counts; `in` checks if text contains something; `.endswith()` checks the ending.

In [14]:
def clean_text(text):
    return " ".join(str(text).split())

for record in records:
    for key in record:
        record[key] = clean_text(record[key])

print(repr(records[6]["instruction"]))
print(repr(records[6]["output"]))
print(repr(records[5]["output"]))

'Write an Instagram caption for a restock.'
'Walnut brownies are back this Saturday. Reserve yours in our DMs.'
''


**What happened:** A loop inside a loop: for every record, for every key, replace the value with its cleaned version. `str(text)` makes sure numbers are turned into text first. Record 5’s output of only spaces became `''` — empty text — which we will catch next.

🔗 **Where you will use this:** The tokenizer turns every space into tokens, so messy spacing wastes training effort and teaches bad habits. On Day 17 you will run checks like these on your real captions.

✏️ **Your turn:**
1. Try `"MILLET Cookies".title()` and `.replace("Rs", "₹")` on some text.
2. Write `clean_text` so it also removes a trailing `!!!` using `.rstrip("!")`.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 7 — Validating records and filtering lists

A record is only useful for training if it has all required fields and none is empty. We write a function that returns a list of problems; an empty list means “valid”. Then a **list comprehension with `if`** keeps only the good records.

In [15]:
REQUIRED = ["instruction", "input", "output"]

def find_problems(record):
    problems = []
    for key in REQUIRED:
        if key not in record:
            problems.append(f"missing '{key}'")
        elif record[key] == "":
            problems.append(f"empty '{key}'")
    return problems

for i, record in enumerate(records):
    problems = find_problems(record)
    if problems:
        print(i, problems)

4 ["missing 'output'"]
5 ["empty 'output'"]


**What happened:** `if problems:` is True only when the list has something in it — an empty list counts as False. Two bad records are found.

In [16]:
valid = [record for record in records if not find_problems(record)]

print(len(records), "->", len(valid), "after removing invalid records")
print([len(r["output"]) for r in valid])

9 -> 7 after removing invalid records
[63, 73, 68, 63, 65, 115, 78]


**What happened:** Read it as: “make a list of `record`, for each record in `records`, but only if it has no problems”. The second comprehension makes a list of output lengths in characters.

🔗 **Where you will use this:** Hugging Face does the same with `ds.filter(lambda ex: len(ex["output"]) > 0)` — keep only examples where a condition is True.

✏️ **Your turn:**
1. Add `"platform"` to `REQUIRED` and re-run. What happens to `valid`?
2. Make a list of only the outputs that contain `"Rs"`.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 8 — Removing duplicates with a set

Duplicate examples make the model over-learn them. A **set** is a collection that keeps only one copy of each value and can check “have I seen this?” instantly. It is written with curly brackets but has no keys.

In [17]:
colours = ["red", "blue", "red", "green", "blue"]
unique = set(colours)

print(sorted(unique))
print(len(colours), "->", len(unique))
print("red" in unique)

['blue', 'green', 'red']
5 -> 3
True


**What happened:** The repeats disappear. Sets have no fixed order, so we use `sorted(...)` to print them in a predictable order.

In [18]:
seen = set()
deduped = []
for record in valid:
    key = (record["instruction"].lower(), record["input"].lower(), record["output"].lower())
    if key in seen:
        print("Duplicate removed:", record["output"][:40])
        continue
    seen.add(key)
    deduped.append(record)

print(len(valid), "->", len(deduped))

Duplicate removed: Crunchy millet cookies, baked fresh this
7 -> 6


**What happened:** For each record we build a **tuple** — a fixed group of values in round brackets — of its three texts, in lowercase so “Restock” and “restock” count as the same. Lists can’t go into a set; tuples can. If the key was seen before, skip the record; otherwise remember it and keep the record.

🔗 **Where you will use this:** Duplicates are especially dangerous across the train/validation split: if the same example is in both, validation scores look better than they really are (“leakage”, Day 17).

✏️ **Your turn:**
1. Add a copy of record 0 with different capital letters to `raw_records`, re-run from Lesson 3, and confirm it is caught.
2. Create a set of all the different `instruction` texts in `deduped` and print how many there are.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 9 — Dataset statistics: min, max, average and sorting

Before training, look at the numbers: how long are the examples? Are some suspiciously short or long? Built-in functions do most of the work. `sorted()` and `max()` accept a `key=` telling them what to compare, written with **lambda**: a tiny one-line function without a name.

In [19]:
word_counts = [len(r["output"].split()) for r in deduped]

print("Words per output:", word_counts)
print("Shortest:", min(word_counts), " Longest:", max(word_counts))
print("Average:", round(sum(word_counts) / len(word_counts), 1))

Words per output: [11, 12, 13, 11, 20, 14]
Shortest: 11  Longest: 20
Average: 13.5


**What happened:** `.split()` turns text into a list of words; `len` counts them. `sum`, `min`, `max` and `round` work on any list of numbers.

In [20]:
longest = max(deduped, key=lambda r: len(r["output"]))
print("Longest:", longest["output"])
print()

for r in sorted(deduped, key=lambda r: len(r["output"])):
    print(len(r["output"]), r["output"][:50])

Longest: For coeliacs it is essential. Many others choose it because it suits them. Either way, our bakes taste great first.

63 Crunchy millet cookies, baked fresh this morning. 
65 Walnut brownies are back this Saturday. Reserve yo
68 6 am in our kitchen. Everything you order is baked
73 Priya says our almond flour brownies beat regular 
78 Lohri special: til-gur cookies, gluten free and fu
115 For coeliacs it is essential. Many others choose i


**What happened:** `lambda r: len(r["output"])` means “given a record r, return the length of its output”. `max` compares records using that number; `sorted` orders them from shortest to longest.

🔗 **Where you will use this:** Models have a maximum input length (`max_length`). Examples longer than it get cut off (truncated). On Day 5 you will measure lengths in **tokens** instead of words to choose that number.

✏️ **Your turn:**
1. Sort from longest to shortest with `sorted(..., reverse=True)`.
2. Find the shortest record with `min(..., key=...)`.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 10 — Shuffling and splitting into train / validation

We keep some examples aside (the **validation set**) that the model never trains on. They show whether the model learned the pattern or just memorised. First shuffle randomly, then cut. A **seed** makes the “random” shuffle repeatable.

In [21]:
import random

numbers = list(range(10))
random.Random(42).shuffle(numbers)
print("seed 42:      ", numbers)

numbers = list(range(10))
random.Random(42).shuffle(numbers)
print("seed 42 again:", numbers)

numbers = list(range(10))
random.Random(7).shuffle(numbers)
print("seed 7:       ", numbers)

seed 42:       [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
seed 42 again: [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
seed 7:        [8, 3, 1, 4, 7, 0, 9, 6, 2, 5]


**What happened:** Same seed → same order every time, on every computer. Different seed → different order. `list(range(10))` makes the list 0 to 9.

In [22]:
def split_data(records, val_fraction=0.2, seed=42):
    shuffled = records[:]
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * val_fraction))
    return shuffled[n_val:], shuffled[:n_val]

train, val = split_data(deduped)
print("train:", len(train), " val:", len(val))
print("val example:", val[0]["input"])

train: 5  val: 1
val example: Product: walnut brownies, Saturday


**What happened:** `records[:]` makes a copy so the original list isn’t reordered. `int(6 * 0.2)` is `int(1.2)` = 1, and `max(1, ...)` guarantees at least one validation example. The function **returns two values**, and `train, val = ...` unpacks them into two variables.

In [23]:
train_outputs = {r["output"] for r in train}
leaks = [r for r in val if r["output"] in train_outputs]
print("Validation examples also in train:", len(leaks))

Validation examples also in train: 0


**What happened:** Curly brackets with `for` make a **set comprehension**. We check that no validation answer also appears in training: zero leaks, thanks to the de-duplication in Lesson 8.

🔗 **Where you will use this:** Hugging Face: `ds.train_test_split(test_size=0.2, seed=42)` does exactly this. Validation loss (Day 7) is calculated on the validation set.

✏️ **Your turn:**
1. Change `seed=42` to `seed=1`. Which example is now in validation?
2. Change `val_fraction` to `0.5`. How many are in each set?

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 11 — Saving the clean data

The cleaned and split data goes back into JSONL files that a training script can read. The `os` library handles folders. `ensure_ascii=False` keeps symbols like ₹ and Hindi text readable in the file instead of turning them into codes.

In [24]:
import os

def save_jsonl(records, path):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_jsonl(train, "data/train.jsonl")
save_jsonl(val, "data/val.jsonl")

print(sorted(os.listdir("data")))
print(len(load_jsonl("data/train.jsonl")), "records in data/train.jsonl")

['train.jsonl', 'val.jsonl']
5 records in data/train.jsonl


**What happened:** `os.path.dirname("data/train.jsonl")` gives the folder part, `"data"`. `os.makedirs(..., exist_ok=True)` creates it and doesn’t complain if it already exists. Reading the file back with our own `load_jsonl` proves the save worked. In Colab, click the folder icon on the left to see the new `data` folder.

🔗 **Where you will use this:** Training scripts save their results the same way: `os.makedirs("outputs", exist_ok=True)`, then checkpoints, logs and adapters go inside it (Day 10).

✏️ **Your turn:**
1. Save `deduped` to `data/all.jsonl` and list the folder again.
2. Open `data/val.jsonl` from the Colab file browser and read it.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 12 — Named arguments and ** unpacking

Fine-tuning code passes many settings to functions. They are given **by name** (`epochs=3`), so order doesn’t matter and the code reads clearly. `**` unpacks a dictionary into named arguments; `**extra` in a function definition collects any extra named arguments into a dictionary.

In [25]:
def describe_run(model, epochs=3, learning_rate=0.0002):
    print(f"Training {model} for {epochs} epochs at lr={learning_rate}")

describe_run("Qwen-0.5B")
describe_run("Qwen-0.5B", epochs=1)
describe_run(learning_rate=0.001, model="SmolLM")

Training Qwen-0.5B for 3 epochs at lr=0.0002
Training Qwen-0.5B for 1 epochs at lr=0.0002
Training SmolLM for 3 epochs at lr=0.001


**What happened:** Missing arguments use their defaults. Named arguments can come in any order.

In [26]:
settings = {"model": "Qwen-0.5B", "epochs": 5, "learning_rate": 0.0001}
describe_run(**settings)

Training Qwen-0.5B for 5 epochs at lr=0.0001


**What happened:** `**settings` is the same as writing `describe_run(model="Qwen-0.5B", epochs=5, learning_rate=0.0001)`. Keys must match the argument names.

In [27]:
def make_settings(max_new_tokens=120, **extra):
    settings = {"max_new_tokens": max_new_tokens}
    settings.update(extra)
    return settings

print(make_settings())
print(make_settings(60, do_sample=True, temperature=0.7))

{'max_new_tokens': 120}
{'max_new_tokens': 60, 'do_sample': True, 'temperature': 0.7}


**What happened:** `**extra` caught `do_sample` and `temperature` into a dictionary. `.update(...)` merges one dictionary into another.

🔗 **Where you will use this:** Day 1’s `model.generate(**inputs, **gen_settings)` unpacks two dictionaries at once. Training configs such as `TrainingArguments(output_dir="out", num_train_epochs=3, learning_rate=2e-4)` are long lists of named arguments.

✏️ **Your turn:**
1. Call `describe_run(**settings, epochs=1)`. Read the error: `epochs` was given twice.
2. Add `"top_p": 0.9` through `make_settings` and print it.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 13 — Classes and objects: settings bundled together

A **class** is a blueprint — like a part drawing. An **object** is one thing made from it — like a part produced from the drawing. Objects hold data (**attributes**, reached with a dot) and actions (**methods**). `__init__` runs when an object is created; `self` means “this particular object”.

In [28]:
class TrainingConfig:
    def __init__(self, model_name, epochs=3, learning_rate=2e-4):
        self.model_name = model_name
        self.epochs = epochs
        self.learning_rate = learning_rate

    def summary(self):
        return f"{self.model_name}: {self.epochs} epochs, lr {self.learning_rate}"

config = TrainingConfig("Qwen/Qwen2.5-0.5B-Instruct", epochs=2)
print(config.epochs)
print(config.summary())

config.epochs = 4
print(config.summary())

2
Qwen/Qwen2.5-0.5B-Instruct: 2 epochs, lr 0.0002
Qwen/Qwen2.5-0.5B-Instruct: 4 epochs, lr 0.0002


**What happened:** `TrainingConfig(...)` creates an object and runs `__init__`, which stores the values on `self`. `config.epochs` reads an attribute; `config.summary()` calls a method. `2e-4` is scientific notation for 0.0002 — you will see it everywhere for learning rates.

In [29]:
from dataclasses import dataclass

@dataclass
class LoraSettings:
    r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

small = LoraSettings()
bigger = LoraSettings(r=16, lora_alpha=32)
print(small)
print(bigger.r)

LoraSettings(r=8, lora_alpha=16, lora_dropout=0.05)
16


**What happened:** `@dataclass` writes the `__init__` for you from the list of fields. `r: int = 8` means “a field named r, a whole number, default 8” (`: int` is just a label for humans). Printing a dataclass shows all its values.

🔗 **Where you will use this:** Hugging Face’s `TrainingArguments` and PEFT’s `LoraConfig` are built exactly like `LoraSettings`: `LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, ...)` (Day 12). `model` and `tokenizer` are objects too — that is why you write `model.generate()`.

✏️ **Your turn:**
1. Add a method `total_steps(self, n_examples)` to `TrainingConfig` that returns `self.epochs * n_examples`.
2. Create `LoraSettings(r=4)` and print it.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 14 — Padding a batch by hand

Models process several examples at once (a **batch**), and the numbers must form a perfect rectangle. Sentences have different lengths, so shorter ones are **padded** with a filler token. An **attention mask** tells the model which positions are real (1) and which are padding (0).

In [30]:
batch = [
    [151644, 872, 198],
    [151644, 872, 198, 2610, 525],
    [151644, 872],
]
PAD_ID = 0
longest = max(len(seq) for seq in batch)

input_ids = []
attention_mask = []
for seq in batch:
    n_pad = longest - len(seq)
    input_ids.append(seq + [PAD_ID] * n_pad)
    attention_mask.append([1] * len(seq) + [0] * n_pad)

for ids, mask in zip(input_ids, attention_mask):
    print(ids, mask)
print("shape:", len(input_ids), "x", len(input_ids[0]))

[151644, 872, 198, 0, 0] [1, 1, 1, 0, 0]
[151644, 872, 198, 2610, 525] [1, 1, 1, 1, 1]
[151644, 872, 0, 0, 0] [1, 1, 0, 0, 0]
shape: 3 x 5


**What happened:** Three token lists of lengths 3, 5 and 2. `[0] * 2` makes `[0, 0]`; `list + list` joins lists. `zip(a, b)` walks two lists side by side. The result is 3 × 5: 3 sequences, 5 positions each — the **shape**.

In [31]:
MAX_LENGTH = 4
truncated = [seq[:MAX_LENGTH] for seq in batch]
print(truncated)

[[151644, 872, 198], [151644, 872, 198, 2610], [151644, 872]]


**What happened:** **Truncation** cuts sequences longer than the limit. The 5-token sequence lost its last token.

🔗 **Where you will use this:** This is precisely what `tokenizer(texts, padding=True, truncation=True, max_length=...)` returns: `input_ids` and `attention_mask` (Day 5). The numbers 151644 and 872 are real Qwen token IDs (the start-of-turn marker and the word “user”).

✏️ **Your turn:**
1. Add a 4th sequence of length 7 and re-run. What is the new shape?
2. Change `MAX_LENGTH` to 2.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 15 — A loop that learns

This is the core of all training, in 15 lines of plain Python. A model with **one** weight `w` learns the price of a box of cookies from examples. Each round: predict, measure the error (**loss**), work out which direction reduces the loss (**gradient**), and nudge `w` that way by a small step (**learning rate**).

In [32]:
data = [(1, 249), (2, 498), (3, 747)]   # (boxes, price in Rs)

def train(learning_rate, epochs, show_every=1):
    w = 0.0
    for epoch in range(epochs):
        loss = 0.0
        grad = 0.0
        for x, y in data:
            prediction = w * x
            error = prediction - y
            loss += error ** 2
            grad += 2 * error * x
        loss /= len(data)
        grad /= len(data)
        w = w - learning_rate * grad
        if epoch % show_every == 0 or epoch == epochs - 1:
            print(f"epoch {epoch}: loss {loss:.2f}, w = {w:.2f}")
    return w

w = train(learning_rate=0.05, epochs=20, show_every=4)
print("Learned price per box:", round(w, 2))

epoch 0: loss 289338.00, w = 116.20
epoch 4: loss 1894.07, w = 238.26
epoch 8: loss 12.40, w = 248.13
epoch 12: loss 0.08, w = 248.93
epoch 16: loss 0.00, w = 248.99
epoch 19: loss 0.00, w = 249.00
Learned price per box: 249.0


**What happened:** `w` starts at 0 (knows nothing). `error ** 2` squares the error so it is always positive and big mistakes count more; the average is the loss. `grad` says how the loss changes if `w` grows: positive means “w is too big”. `w - learning_rate * grad` moves `w` the opposite way. `loss += x` is short for `loss = loss + x`. `%` gives the remainder, so `epoch % 4 == 0` is true every 4th epoch. The loss falls toward 0 and `w` reaches 249.

In [33]:
w = train(learning_rate=0.25, epochs=6)

epoch 0: loss 289338.00, w = 581.00
epoch 1: loss 514378.67, w = -193.67
epoch 2: loss 914450.96, w = 839.22
epoch 3: loss 1625690.60, w = -537.96
epoch 4: loss 2890116.62, w = 1298.28
epoch 5: loss 5137985.11, w = -1150.05


**What happened:** Same code, learning rate 5× bigger. Each step overshoots the answer by more than the last, `w` swings wildly and the loss explodes. This is the classic sign of a learning rate that is too high — you will see it as `NaN` or exploding loss in real training (Day 18).

🔗 **Where you will use this:** A real model has 494 million weights instead of one. `loss.backward()` calculates the gradient for every weight automatically, and `optimizer.step()` does `w = w - learning_rate * grad` for all of them (Days 6 and 8).

✏️ **Your turn:**
1. Try `learning_rate=0.01`. Does it reach 249 within 20 epochs?
2. Change the data to `(1, 199), (2, 398)` and train again.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Lesson 16 — Mini project: the whole pipeline in one function

Everything from this lesson, combined into one reusable tool: load → clean → validate → de-duplicate → shuffle and split → save → report. The same code is saved as `prepare_dataset.py` in this folder, so you can run it on your own files.

In [34]:
import json
import os
import random

REQUIRED = ["instruction", "input", "output"]


def load_jsonl(path):
    """Read one JSON record per line. Skip blank or broken lines instead of crashing."""
    records = []
    with open(path, encoding="utf-8") as f:
        for number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Skipped line {number}: not valid JSON")
    return records


def clean_text(text):
    """Remove spaces at the ends and collapse repeated spaces inside."""
    return " ".join(str(text).split())


def find_problems(record):
    """Return a list of problems with one record. An empty list means it is valid."""
    problems = []
    for key in REQUIRED:
        if key not in record:
            problems.append(f"missing '{key}'")
        elif record[key] == "":
            problems.append(f"empty '{key}'")
    return problems


def save_jsonl(records, path):
    """Write records one per line, creating the folder if needed."""
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def prepare(raw_path, out_dir="data", val_fraction=0.2, seed=42):
    """Run the whole cleaning pipeline and return a report dictionary."""
    report = {}

    records = load_jsonl(raw_path)
    report["loaded"] = len(records)

    for record in records:
        for key in record:
            record[key] = clean_text(record[key])

    valid = [r for r in records if not find_problems(r)]
    report["invalid_removed"] = len(records) - len(valid)

    seen = set()
    deduped = []
    for r in valid:
        key = (r["instruction"].lower(), r["input"].lower(), r["output"].lower())
        if key not in seen:
            seen.add(key)
            deduped.append(r)
    report["duplicates_removed"] = len(valid) - len(deduped)

    shuffled = deduped[:]
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * val_fraction))
    val, train = shuffled[:n_val], shuffled[n_val:]

    save_jsonl(train, os.path.join(out_dir, "train.jsonl"))
    save_jsonl(val, os.path.join(out_dir, "val.jsonl"))
    report["train"] = len(train)
    report["val"] = len(val)
    return report


report = prepare("raw_captions.jsonl", out_dir="data_final")
for key, value in report.items():
    print(f"{key}: {value}")

Skipped line 10: not valid JSON
loaded: 9
invalid_removed: 2
duplicates_removed: 1
train: 5
val: 1


**What happened:** One function call runs the whole pipeline, starting again from the raw file. `report` is a dictionary, and `.items()` gives each key–value pair for printing. From 10 raw lines (one of them broken) we get 6 clean, unique examples: 5 for training and 1 for validation.

🔗 **Where you will use this:** On Day 4 you will run `python prepare_dataset.py my_captions.jsonl` on a dataset you write, then feed `data/train.jsonl` into training.

✏️ **Your turn:**
1. Add 5 of your own caption examples to `raw_records` (Lesson 3) and re-run the notebook from the top.
2. Run the script version in Colab: upload `prepare_dataset.py` and `raw_captions.jsonl`, then run `!python prepare_dataset.py`.

In [ ]:
# Try it here. Type your code below and press Shift + Enter.


---
## Bridge: code you will meet later in the course

| Real fine-tuning code | Lesson | Course day | In plain English |
|---|---|---|---|
| `with open("train.jsonl", encoding="utf-8") as f:` | 1 | Day 4 | Open a dataset file. |
| `json.loads(line)  /  json.dumps(record)` | 2, 3 | Day 4 | Text ↔ dictionary. |
| `load_dataset("json", data_files="train.jsonl")` | 3 | Day 4 | Hugging Face’s version of `load_jsonl`. |
| `except torch.cuda.OutOfMemoryError:` | 4 | Day 18 | Catch a GPU memory error. |
| `ds[0]  /  ds.column_names` | 5 | Day 4 | Look inside a dataset. |
| `ds.filter(lambda ex: len(ex["output"]) > 0)` | 7, 9 | Day 17 | Keep only valid examples. |
| `ds.train_test_split(test_size=0.2, seed=42)` | 10 | Day 4 | Shuffle and split. |
| `os.makedirs("outputs", exist_ok=True)` | 11 | Day 10 | Create a folder for results. |
| `model.generate(**inputs, **gen_settings)` | 12 | Day 1–2 | Unpack dictionaries into arguments. |
| `TrainingArguments(output_dir="out", num_train_epochs=3, learning_rate=2e-4)` | 12, 13 | Day 7, 9 | A settings object built from named arguments. |
| `LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05)` | 13 | Day 12 | A dataclass of LoRA settings. |
| `tokenizer(texts, padding=True, truncation=True, max_length=512)` | 14 | Day 5 | Pad and cut a batch; returns input_ids + attention_mask. |
| `for step, batch in enumerate(train_loader):` | 1, 15 | Day 8 | Loop over batches with a counter. |
| `loss.backward()` | 15 | Day 6, 8 | Compute the gradient for every weight. |
| `optimizer.step()` | 15 | Day 6, 8 | w = w − learning_rate × gradient, for all weights. |

## Checkpoint ✅
You have finished Day 0C when you can explain in your own words:
1. Why `load_jsonl` uses `try/except`, and what `continue` does.
2. What `.get("output", "MISSING")` returns when the key is missing.
3. Why we store the duplicate check as a tuple inside a set.
4. Why the same seed always gives the same train/validation split.
5. What the attention mask `[1, 1, 1, 0, 0]` tells the model.
6. What happened when the learning rate was too high in Lesson 15.

**Next:** Day 1 — run the real Qwen model. Then Day 4 will use `prepare_dataset.py` on a dataset you write yourself.